## Cifratura simmetrica e cifrario a sostituzione
La cifratura trasforma un msg leggibile, detto **plaintext P**, in un testo cifrato detto **ciphertext C**. La trasformazione deve esser reversibile solo da chi possiede la chiave:
$$C = Encrypt(K, P) \quad P = Decrypt(K, C)$$

Nella **Cifratura Simmetrica** si usa la stessa chiave segreta per cifrare e decifrare. L'algoritmo al contrario può essere pubblico: la sicurezza deve dipendere dalla chiave e non dal fatto che qualcuno ignori come funziona l'algoritmo. 

Si mostra un esempio di cifrario a sostituzione: ogni lettera viene sempre sostituita con la stessa lettera. Il problema di questa cifratura è che la sostituzione conserva molte caratteristiche del testo originale (es. lettere frequenti restano frequenti anche se cambiano simbolo, le doppie restano doppie, parole ripetute producono la stessa sequenza di simboli etc..).  
Per questo motivo si può attaccare il cifrario con analisi delle frequenze e tecniche banali di crittoanalisi per risalire al testo originale.

Quindi rendere il contenuto illegibile a prima vista non basta, le regolarità che restano nel testo cifrato possono rivelare informazioni.

### One-time pad o Cifrario di Vernam
Il one-time pad cifra una sequenza di bit $M$ usando una chiave $K$ della stessa lunghezza. L'operazone è lo **XOR** (1 se e solo se i due bit sono diversi), indicato con $\oplus$:
$$C = M \oplus K$$
Per decifrare si applica nuovamente lo xor con la stessa chiave, e poiché $K \oplus K = 0$, si ottiene:
$$C \oplus K = M \oplus K \oplus K = M$$
Si dice che il one-time pad offre **segretezza perfetta**. Infatti supponiamo che un bit della chiave sia **davvero casuale** (i.e. vale 1 o 0 w.p. 0.5). Allora il bit cifrato non può aiutarci in alcun modo a capire quale fosse il bit originale. Quindi anche dopo aver osservato il ciphertext, la probabilità di indovinare il plaintext è la stessa che avevamo prima. 

Questa proprietà è appunto nota come **segretezza perfetta**: osservare il ciphertext non fornisce alcuna informazione sul messaggio originale. Vale però soltanto se la chiave:
1. è veramente casuale
2. è lunga almeno quanto il messaggio
3. viene usata **una sola volta**

La terza condizione è cruciale: se cifro due messaggi con la stessa chiave $C_1 = M_1 \oplus K$ e $C_2 = M_2 \oplus K$, allora posso calcolare:
$$C_1 \oplus C_2 = M_1 \oplus K \oplus M_2 \oplus K = M_1 \oplus M_2$$
per cui la chiave "scompare". Se poi l'attaccante conosce anche $M_1$, allora può ricavare direttamente $M_2$:
$$M_2 = M_1 \oplus C_1 \oplus C_2$$
Il one-time pad inoltre è **malleabile**: un attaccante può modificare dei bit del ciphertext e causare modifiche prevedibili nei corrispondenti bit del plaintext decifrato. Quindi la segretezza perfetta **non garantisce l'integrità del messaggio**: chi riceve non sa, grazie alla sola cifratura, se qualcuno lo abbia alterato. 

Quindi il one-time pad ha una sicurezza teorica perfetta, ma è poco pratico in quanto distribuire e conservare una chiave segreta casuale lunga quanto ogni messaggio è difficile. Servirebbero inoltre meccanismi distinti per verificare l'integrità.


### Numeri casuali e pseudocasuali
Un **TRNG** (True Random Number Generator) genera numeri casuali basandosi su fenomeni fisici, come il rumore termico o la radioattività.  
Un **PRNG** (Pseudo Random Number Generator) invece è un algoritmo: a partire da un seme iniziale genera una sequenza di numeri che sembrano casuali, ma sono deterministici. La funzione ```rand()``` in C è un esempio di PRNG e non va quindi assunta come generatore crittograficamente sicuro.

Per la crittografia non basta che la sequenza *sembri* casuale in un test statistico: deve anche essere **difficile da prevedere**, persino per chi conosce l'algoritmo. Si usano quindi generatori progettati per la crittografia, detti **CSPRNG** (Cryptographically Secure Pseudo Random Number Generator). 

Vediamo un esempio di attacco di questo tipo con l'attacco al protocollo RFID (Radio Frequency IDentification). In questo protocollo, un lettore invia un numero casuale a un tag, che risponde con un messaggio cifrato usando una chiave segreta condivisa. Tag e lettore in particolare condividon quindi un segreto $S$ e uno stato $K_i$ che viene aggiornato ad ogni sessione. 

Il protocollo invia due valori:
$$M_1 = S \oplus K_i \quad M_2 = S \oplus K_{i+1} \quad K_{i+1} = PRNG(K_i)$$
Un osservatore potrebbe intercettare $M_1$ e $M_2$. Facendo lo xor eliminerebbe il segreto $S$, ottenendo $M_1 \oplus M_2 = K_i \oplus PRNG(K_i)$.  
Ma se il PRNG è noto, l'attaccante può provare i possibili stati $x$ e cercare quelli per cui $x \oplus PRNG(x) = M_1 \oplus M_2$. Se il segreto è di soli 16 bit, gli stati possibili sono $2^{16} = 65536$, quindi l'attaccante può provare tutti i valori in pochi secondi. Trovato lo stato compatibile K_i, può calcolare il segreto $S = M_1 \oplus K_i$ e quindi rispondere al tag come se fosse il lettore legittimo, nonché calcolare i successivi stati $K_{i+1} = PRNG(K_i)$ e quindi intercettare le future comunicazioni.

In realtà una sola osservazione potrebbe dare più stati candidati $x$, ma solo uno effettivamente corrispondente a $K_i$. Ciò non risolve il problema in quanto ulteriori intercettazioni permettono di eliminare le ambiguità.

### Sicurezza IND-CPA / Semantic Security
Dire che un cifrario "nasconde bene il messaggio" è troppo vago. Bisogna specificare **cosa può fare l'attaccante e che cosa potrebbe riuscire a scoprire**.

Si introduce il concetto di **IND-CPA** (Indistinguishability under Chosen Plaintext Attack). L'attaccante può richiedere la cifratura di messaggi a sua scelta e ricevere i corrispondenti ciphertext. Si esprime la definizione tramite un gioco:
1. L'attaccante sceglie due messaggi $M_0$ e $M_1$ della stessa lunghezza.
2. Il cifrario sceglie un bit casuale $b \in \{0,1\}$ e cifra $M_b$
3. L'attaccante riceve il ciphertext $C = Enc(K, M_b)$ ($K$ è la chiave di cifratura) e deve indovinare $b$.

La stessa lunghezza evita che l'attaccante possa distinguere i due messaggi semplicemente osservando la lunghezza del ciphertext. Un cifrario IND-CPA sicuro garantisce che un attaccante con risorse computazionali realistiche non possa indovinare **sensibilmente meglio del caso**, ossia w.p. maggiore di 0.5 più una quantità trascurabile. 

L'accesso ai messaggi cifrati conta tantissimo perché garantisce che ogni cifrario IND-CPA sicuro utilizzi adeguatamente casualità o comunque un nonce che cambi ad ogni cifratura. Se infatti si usasse un cifrario deterministico per cui con la stessa chiave viene sempre dato lo stesso risultato, allora l'attaccante potrebbe sapere il risultato di un messaggio a priori e chiedendo di cifrare quel messaggio tra i due come $M_0$ capirebbe subito quale è stato cifrato (se gli esce il risultato che conosce allora $M_0$, altrimenti $M_1$). Quindi un cifrario deterministico non può essere IND-CPA sicuro (**consequence 1: encryption MUST be randomized**).  
La seconda conseguenza è che **if a substring repeats, it must encrypt to a different ciphertext**. Infatti se l'attaccante scrive CIAOCIAOCIAO come messaggio e vede che il ciphertext è una sequenza di simboli ma ripetuta tre volte allora può capire che il messaggio di origine era proprio CIAOCIAOCIAO. 

Quindi ad esempio SHA-256 non è IND-CPA sicuro (anche se, attenzione, SHA-256 non è un cifrario ma una funzione di hash. Un cifrario è un sistema per nascondere un msg e permettere a chi ha la chiave di recuperlarlo, quindi ha Encrypt e Decrypt. Invece SHA_256 calcola un'impronta i lunghezza fissa.)

In definitiva quindi con **Semantic Security** una garanzia di sicurezza per cui un attaccante efficiente non è in grado di ricavare informazioni utili sul contenuto del messaggio cifrato che non potesse già ricavare senza averlo osservato. IND-CPA rende questa idea verificabile con un "gioco formale". 

In realtà nella formulazione rigorosa si ammette che l'avversario possa indovinare il messaggio con probabilità pari a $0.5 + \epsilon$ per un $\epsilon$ trascurabile.  
Sarebbe a dire che **per ogni** funzione **efficiente** realizzabile dall'avversario $\phi$ sul ciphertext $\phi: ciphertext \to \{0,1\}$, si ha che:
$$\Pr[\phi(Enc(k, m_0))] - \Pr[\phi(Enc(k, m_1))] \leq \epsilon$$
dove con funzione efficiente si intende che l'avversario non può fare calcoli esponenziali o comunque troppo costosi. 

Al contrario, la definizione rigorosa di **segretezza perfetta** fa si che **per ogni funzione** $\phi$ (anche non efficiente) realizzabile dall'avversario, si abbia:
$$\Pr[\phi(Enc(k, m_0))] = \Pr[\phi(Enc(k, m_1))]$$